**Word embeddings and vector space model based classification and evaluation**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
import nltk


In [ ]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
newsgroups_data = fetch_20newsgroups(subset='all', categories=['comp.graphics', 'sci.med'])
X, y = newsgroups_data.data, newsgroups_data.target

# **Tokenization**

In [ ]:
X_tokenized = [word_tokenize(text.lower()) for text in X]

# **Word2Vec**

In [ ]:
w2v_model = Word2Vec(sentences=X_tokenized, vector_size=100, window=5, min_count=2, workers=4)
w2v_model.train(X_tokenized, total_examples=len(X_tokenized), epochs=10)


(5095441, 7233590)

In [ ]:
def compute_avg_w2v_vectors(texts, model, vector_size):
    vectors = []
    for text in texts:
        word_vectors = [model.wv[word] for word in text if word in model.wv]
        if word_vectors:
            vectors.append(np.mean(word_vectors, axis=0))
        else:
            vectors.append(np.zeros(vector_size))
    return np.array(vectors)

X_w2v = compute_avg_w2v_vectors(X_tokenized, w2v_model, 100)

#**TF-IDF**

In [ ]:
tfidf_vectorizer = TfidfVectorizer(max_features=1000)
X_tfidf = tfidf_vectorizer.fit_transform(X).toarray()

#**Training both classifiers**

In [ ]:
X_train_w2v, X_test_w2v, y_train, y_test = train_test_split(X_w2v, y, test_size=0.2, random_state=42)
X_train_tfidf, X_test_tfidf, _, _ = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)


#**Logistic Regression on Word2Vec features**

In [ ]:
clf_w2v = LogisticRegression(max_iter=1000)
clf_w2v.fit(X_train_w2v, y_train)
y_pred_w2v = clf_w2v.predict(X_test_w2v)
accuracy_w2v = accuracy_score(y_test, y_pred_w2v)

#**Logistic Regression on TF-IDF features**

In [ ]:
clf_tfidf = LogisticRegression(max_iter=1000)
clf_tfidf.fit(X_train_tfidf, y_train)
y_pred_tfidf = clf_tfidf.predict(X_test_tfidf)
accuracy_tfidf = accuracy_score(y_test, y_pred_tfidf)

#**Evaluation and Comparison**

In [ ]:
print(f'Accuracy with Word2Vec features: {accuracy_w2v:.4f}')
print(f'Accuracy with TF-IDF features: {accuracy_tfidf:.4f}')

Accuracy with Word2Vec features: 0.9567
Accuracy with TF-IDF features: 0.9440


Therefore we can observe that Word2Vec gives more accuracy